In [2]:
import pandas as pd
import string
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
# import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.pipeline import Pipeline

In [3]:
# df = pd.read_csv('/kaggle/input/datasets/omar2716/emails-dataset/spam_and_ham_classification.csv')
df= pd.read_csv('spam_and_ham_classification.csv')

In [4]:
df

,label,text
0,ham,into the kingdom of god and those that are ent...
1,spam,there was flow at hpl meter 1505 on april firs...
2,ham,take a look at this one campaign for bvyhprice...
3,spam,somu wrote actually thats what i was looking f...
4,spam,fathi boudra wrote i fixed the issue in the sv...
...,...,...
9984,ham,this would be a great tragedy for all concerne...
9985,ham,"hello , welcome to medzonline filamentous shop..."
9986,ham,this is amazing stuff add some inches fast saf...
9987,spam,author jra date escapenumber escapenumber esca...


In [5]:
df.head()

,label,text
0,ham,into the kingdom of god and those that are ent...
1,spam,there was flow at hpl meter 1505 on april firs...
2,ham,take a look at this one campaign for bvyhprice...
3,spam,somu wrote actually thats what i was looking f...
4,spam,fathi boudra wrote i fixed the issue in the sv...


In [6]:
df.tail()

,label,text
9984,ham,this would be a great tragedy for all concerne...
9985,ham,"hello , welcome to medzonline filamentous shop..."
9986,ham,this is amazing stuff add some inches fast saf...
9987,spam,author jra date escapenumber escapenumber esca...
9988,ham,anatrim escapenumber the newest and most attra...


In [7]:
df.shape

(9989, 2)

In [8]:
df.size

19978

In [9]:
df.sample(20)

,label,text
8602,ham,christmass s @ | e - w ! ndows xp home\nwe hav...
8707,ham,"5 , meridian east\nleicester le 3 2 wz\nleices..."
3788,spam,subscribe change profile contact us long term ...
1057,spam,hi the silverman's paper introduction offer ho...
7491,spam,to : all enron employees :\nthis is a reminder...
462,spam,?\n- - - - - original message - - - - -\nfrom ...
2334,ham,youcansave . com\n4330 commerce drive\nbatavia...
7718,spam,"dear richard ,\nthanks for your message - i ju..."
4032,ham,our warmest greetings unique proposal for you ...
112,ham,dear henna plg uwaterloo ca http kkeioop com w...


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9989 entries, 0 to 9988
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   9989 non-null   object
 1   text    9989 non-null   object
dtypes: object(2)
memory usage: 156.2+ KB


In [11]:
df = df.dropna()
all_words = ' '.join(df.text).split()

In [12]:
len(all_words)

2747483

In [13]:
count = Counter(all_words)
count

Counter({'escapenumber': 133208,
         'the': 85264,
         'to': 56587,
         '.': 52672,
         'and': 42360,
         '-': 40970,
         'of': 40427,
         ',': 39261,
         'a': 36859,
         'in': 28900,
         'escapelong': 24634,
         'for': 22671,
         'you': 22222,
         'is': 21731,
         'i': 19917,
         '/': 19489,
         'that': 17078,
         ':': 16702,
         'this': 16485,
         'on': 16397,
         'with': 13249,
         'it': 13230,
         'be': 12177,
         'your': 12109,
         'as': 11360,
         'from': 10399,
         'are': 10055,
         'have': 10014,
         'http': 9844,
         'at': 9395,
         'not': 9285,
         'or': 9107,
         'we': 9072,
         'com': 8889,
         'will': 8633,
         'by': 8574,
         'if': 7868,
         "'": 7816,
         's': 7406,
         'our': 6959,
         'all': 6834,
         'r': 6458,
         'but': 5908,
         ')': 5871,
         'an':

In [14]:
df.label.info()

<class 'pandas.core.series.Series'>
RangeIndex: 9989 entries, 0 to 9988
Series name: label
Non-Null Count  Dtype 
--------------  ----- 
9989 non-null   object
dtypes: object(1)
memory usage: 78.2+ KB


In [15]:
df.describe()

,label,text
count,9989,9989
unique,2,9989
top,ham,into the kingdom of god and those that are ent...
freq,5294,1


In [16]:
df.label.value_counts()

label
ham     5294
spam    4695
Name: count, dtype: int64

> Cleaning Data

In [17]:
df.isna().sum()

label    0
text     0
dtype: int64

In [18]:
df.duplicated().sum()

0

In [19]:
df = df.drop_duplicates()
df = df.dropna()

# حذف علامات الترقيم

In [20]:
punc = string.punctuation

In [21]:
punTable = str.maketrans('','',punc)                                            

In [22]:
df.text = df.text.str.translate(punTable)

# حروف صغيرة

In [23]:
df.text = df.text.str.lower()

# حذف الكلمات الشائعة

In [24]:
# nltk.download('stopwords')
# nltk.download('punkt')
stopWords = set(stopwords.words('english'))

In [25]:
df.text = df.text.str.split()

In [26]:
df

,label,text
0,ham,"[into, the, kingdom, of, god, and, those, that..."
1,spam,"[there, was, flow, at, hpl, meter, 1505, on, a..."
2,ham,"[take, a, look, at, this, one, campaign, for, ..."
3,spam,"[somu, wrote, actually, thats, what, i, was, l..."
4,spam,"[fathi, boudra, wrote, i, fixed, the, issue, i..."
...,...,...
9984,ham,"[this, would, be, a, great, tragedy, for, all,..."
9985,ham,"[hello, welcome, to, medzonline, filamentous, ..."
9986,ham,"[this, is, amazing, stuff, add, some, inches, ..."
9987,spam,"[author, jra, date, escapenumber, escapenumber..."


In [27]:
df.text = df.text.apply(lambda x:[word for word in x if word not in stopWords])

In [28]:
df

,label,text
0,ham,"[kingdom, god, entering, lord, pardon, escapen..."
1,spam,"[flow, hpl, meter, 1505, april, first, deal, t..."
2,ham,"[take, look, one, campaign, bvyhprice, escapen..."
3,spam,"[somu, wrote, actually, thats, looking, l, r, ..."
4,spam,"[fathi, boudra, wrote, fixed, issue, svn, repo..."
...,...,...
9984,ham,"[would, great, tragedy, concerned, situation, ..."
9985,ham,"[hello, welcome, medzonline, filamentous, shop..."
9986,ham,"[amazing, stuff, add, inches, fast, safe, effe..."
9987,spam,"[author, jra, date, escapenumber, escapenumber..."


In [29]:
df.text= df.text.apply(lambda x:' '.join([word for word in x if len(word) > 2]))

In [30]:
df

,label,text
0,ham,kingdom god entering lord pardon escapenumber ...
1,spam,flow hpl meter 1505 april first deal ticket de...
2,ham,take look one campaign bvyhprice escapenumber ...
3,spam,somu wrote actually thats looking user entered...
4,spam,fathi boudra wrote fixed issue svn repo rev es...
...,...,...
9984,ham,would great tragedy concerned situation digiti...
9985,ham,hello welcome medzonline filamentous shop plea...
9986,ham,amazing stuff add inches fast safe effective s...
9987,spam,author jra date escapenumber escapenumber esca...


# تحويل الكلمات إلى جذورها الأساسية أو صيغتها المعيارية.

In [31]:
# nlp = spacy.load("en_core_web_sm")
# df['text'] = df['text'].apply(lambda x: " ".join([token.lemma_ for token in nlp(str(x))]))

# تقسيم البيانات

In [32]:
x = df.text
y = df.label

In [33]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y) # stratify=y  يضمن ان النموذج يوازن بين البيانات بدون تحيز

In [34]:
tfidf=TfidfVectorizer()

In [49]:
x_train_vec = tfidf.fit_transform(x_train) #fit تبني القاموس transform تطبق القاموس fit_transform تعمل العملين 
x_test_vec = tfidf.transform(x_test)

# pipeline

In [56]:
model_pipeline = Pipeline([("tfidf",TfidfVectorizer(lowercase=True,stop_words="english",max_features=5000,)),("model",LinearSVC())])
model_pipeline.fit(x_train,y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, stop_words='english')),
                ('model', LinearSVC())])

# بنا النموذج

In [36]:
model = MultinomialNB()

In [37]:
model.fit(x_train_vec,y_train)

MultinomialNB()

In [38]:
pred = model.predict(x_test_vec)

In [39]:
pred

array(['spam', 'spam', 'spam', ..., 'ham', 'spam', 'spam'], dtype='<U4')

In [40]:
acc = accuracy_score(pred,y_test)

In [41]:
acc

0.970970970970971

In [42]:
classification = classification_report(pred,y_test)

In [43]:
print(classification)

              precision    recall  f1-score   support

         ham       0.96      0.98      0.97      1035
        spam       0.98      0.96      0.97       963

    accuracy                           0.97      1998
   macro avg       0.97      0.97      0.97      1998
weighted avg       0.97      0.97      0.97      1998



In [44]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

In [45]:
model = LinearSVC()

In [46]:
model.fit(x_train_vec,y_train)

LinearSVC()

In [47]:
pred = model.predict(x_test_vec)

In [48]:
acc = accuracy_score(pred,y_test)
acc

0.9824824824824825